# Is MoMA's increasing representation of women artists driven by acquiring their work in real time, or by retroactively "correcting" the collection, i.e., acquiring older, previously-overlooked work by women decades after it was made?

In [11]:
# Import Libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import json

In [12]:
df = pd.read_csv('Artworks.csv', encoding = 'utf-8-sig', low_memory = False)
df.shape

(160705, 30)

In [13]:
from pandas.tseries.offsets import YearBegin
import re
def split_parenthesized(value):
  if pd.isna(value):
    return[]
  s = value.strip()
  if s.startswith('(') and s.endswith(')'):
    s = s[1:-1]
    return re.split(r'\)\s*\(',s)
def first_or_nan(lst):
  if not lst or lst[0] == '':
    return np.nan
  return lst[0]
def extract_year(value):
  if pd.isna(value):
    return np.nan
  s = str(value)
  match = re.search(r'\d{4}', s)
  if not match:
      return np.nan
  year = int(match.group())
  if re.search(r'B\.?C\.?', s, re.IGNORECASE):
      year = -year
  return year

df['Gender_list'] = df['Gender'].apply(split_parenthesized)
df['Gender_primary'] = df['Gender_list'].apply(first_or_nan)

df['Year'] = df['Date'].apply(extract_year)
df['DateAcquired'] = pd.to_datetime(df['DateAcquired'],
                                    errors = 'coerce')
df['AcquisitionYear'] = df['DateAcquired'].dt.year
df['Lag'] = df['AcquisitionYear'] - df['Year']

subset = df[df['Gender_primary'].isin(['male','female']) &
            df['Lag'].notna()]
subset.groupby('Gender_primary')['Lag'].describe()


,count,mean,std,min,25%,50%,75%,max
Gender_primary,,,,,,,,
female,20308.0,22.894869,25.408002,-10.0,3.0,11.0,40.0,166.0
male,122197.0,28.611652,28.127021,-14.0,5.0,21.0,43.0,196.0


In [14]:
print('Year missing:', df['Year'].isna().sum(), '/', len(df))
print('AcquisitionYear missing:',
      df['AcquisitionYear'].isna().sum(),'/', len(df))
print('Lag missing:', df['Lag'].isna().sum(),'/', len(df))
print()
print(df['Gender_primary'].value_counts(dropna=False).head(10))
print()
print('subset rows:', len(subset))

Year missing: 4423 / 160705
AcquisitionYear missing: 5468 / 160705
Lag missing: 9453 / 160705

Gender_primary
male                     129263
female                    21475
NaN                        9847
female (transwoman)          62
non-binary                   42
male (trans? ftm?)           11
gender non-conforming         2
woman, non-binary             2
transgender woman             1
Name: count, dtype: int64

subset rows: 142505


In [15]:
print(df['Date'].dtype)
print(df['Date'].head(10).tolist())
print()
print(extract_year(df['Date'].iloc[0]))
print(extract_year('1973'))

object
['1896', '1987', '1903', '1980', '1903', '1976-77', '1976-77', '1976-77', '1976-77', '1976-77']

1896
1973


In [16]:
neg_lag = subset[subset['Lag']<0]
print('Rows with negative lag:', len(neg_lag))
print(neg_lag['Gender_primary'].value_counts())
print()
neg_lag[['Title','Artist','Date','Year','DateAcquired','AcquisitionYear','Lag']].head()

Rows with negative lag: 93
Gender_primary
male      77
female    16
Name: count, dtype: int64



,Title,Artist,Date,Year,DateAcquired,AcquisitionYear,Lag
374,"First Unitarian Church and School, Rochester, ...",Louis I. Kahn,1969,1969.0,1967-04-11,1967.0,-2.0
438,"Les Eschelles du Baroque Apartment Building, P...",Ricardo Bofill,1987,1987.0,1985-11-18,1985.0,-2.0
1505,Expandable Containers,William N. Touzani,1987,1987.0,1986-05-16,1986.0,-1.0
4062,Help Prevent Despair. Care,George Delany,1987,1987.0,1986-05-16,1986.0,-1.0
4284,If You are Suffering from Scabies,Abram Games,c. 1944,1944.0,1943-05-12,1943.0,-1.0


In [17]:
subset = subset.copy()
subset['AcquisitionDecade'] = (subset['AcquisitionYear']//10) *10
subset['AcquisitionDecade'] = subset['AcquisitionDecade'].astype('Int64')
x = subset.groupby(['AcquisitionDecade',
                    'Gender_primary'])['Lag'].mean().unstack().round(1)
x

Gender_primary,female,male
AcquisitionDecade,,
1920,NaN,9.0
1930,9.3,13.5
1940,11.4,18.1
1950,15.3,20.9
1960,22.5,28.2
1970,20.9,20.5
1980,14.9,26.8
1990,17.4,25.3
2000,20.0,31.0


In [18]:
bins = list(range(-20, 105, 5))
hist_data = {}
for (decade, gender), group in subset.groupby(['AcquisitionDecade', 'Gender_primary']):
    clipped = group['Lag'].dropna().clip(bins[0], bins[-1] - 1)
    counts, _ = np.histogram(clipped, bins=bins)
    hist_data[f'{int(decade)}_{gender}'] = counts.tolist()

print(json.dumps({'bins': bins, 'data': hist_data}))

{"bins": [-20, -15, -10, -5, 0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80, 85, 90, 95, 100], "data": {"1920_male": [0, 0, 0, 0, 1, 4, 3, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], "1930_female": [0, 0, 0, 0, 22, 7, 7, 4, 1, 1, 0, 2, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0], "1930_male": [0, 1, 0, 0, 548, 292, 172, 124, 224, 82, 7, 38, 26, 16, 47, 3, 0, 3, 10, 6, 1, 0, 3, 0], "1940_female": [0, 0, 0, 0, 190, 183, 74, 29, 15, 15, 6, 58, 15, 0, 2, 0, 1, 0, 2, 0, 0, 0, 0, 0], "1940_male": [0, 0, 0, 2, 1808, 726, 844, 649, 592, 453, 303, 241, 140, 93, 132, 46, 38, 14, 65, 103, 49, 5, 2, 9], "1950_female": [0, 0, 1, 0, 140, 43, 8, 7, 69, 24, 26, 11, 5, 3, 0, 7, 3, 0, 0, 0, 2, 2, 0, 0], "1950_male": [0, 0, 1, 5, 1990, 657, 331, 185, 254, 294, 505, 423, 285, 158, 159, 205, 173, 74, 12, 18, 7, 4, 2, 13], "1960_female": [0, 0, 0, 0, 378, 203, 41, 20, 33, 35, 57, 60, 35, 3, 6, 17, 20, 165, 5, 1, 0, 0, 1, 6], "1960_male": [0, 0, 0, 4, 6631, 4110, 3655, 4111, 5184, 1668, 278

In [19]:
with open('lag_histogram.json', 'w') as f:
    json.dump({'bins': bins, 'data': hist_data}, f)

from google.colab import files
files.download('lag_histogram.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Conclusion

**Research question:** Is MoMA's increasing representation of women artists driven by acquiring their work in real time, or by retroactively "correcting" the collection, i.e., acquiring older, previously-overlooked work by women decades after it was made?

**Finding:** The data points away from retroactive correction. Of the 160,705 artworks in the cleaned dataset, 142,505 have both a usable creation year, acquisition date, and a gender recorded as exactly "male" or "female." Within that group, women's work is acquired sooner after it is made than men's work, not later. The average gap is about 5.7 years (22.9 years for women vs. 28.6 years for men). This holds true in almost every decade from the 1930s to the 2020s. Only the 1970s is close to even. If MoMA were mainly catching up on overlooked women artists years after the fact, we would expect the opposite pattern, a longer wait for women, not a shorter one.

**Nuance:** Acquisition lag has grown for both men and women over time. In the 1920s and 1930s, MoMA acquired art within about 10 years of it being made. Today that gap is 30 to 50 years for both genders. The gender difference sits on top of this larger trend, it does not replace it.

**Limitations:** 18,200 artworks are excluded from this analysis, either because the creation year or acquisition date could not be determined, or because gender was missing or recorded as something other than exactly "male" or "female." Among the artworks that are included, men outnumber women roughly 6 to 1, and even more so before 1960. The 1920s has no acquisitions by women at all. A small number of records, 93 out of 142,505, show an artwork acquired slightly before its listed creation date, almost always because the creation date itself is approximate.

**What this doesn't prove:** This shows a pattern, not a cause. We don't know why the gap exists; it could reflect deliberate MoMA acquisition strategy, market dynamics (contemporary women artists may be more visible/collectible sooner than in past decades), differences in which works get gender-attributed in the dataset, or something else entirely. This analysis can't distinguish between those explanations.